# Xarray-Spatial Viewshed: Line-of-sight visibility from observer points

Viewshed analysis determines which cells in a DEM are visible from a given observer location. Given a point and an observer height, it traces lines of sight across the elevation surface and marks each cell as visible or hidden. This is used in avalanche monitoring, telecommunications tower placement, and landscape planning.

### What you'll build

1. Compute viewshed from an observer on a synthetic Gaussian peak
2. Visualize visible vs. hidden terrain with hillshade overlay
3. Run viewshed on generated mountain terrain with observer elevation
4. Identify blind spots behind ridgelines

![Viewshed preview](images/viewshed_preview.png)

**Jump to a section:**
[Simple viewshed](#Simple-viewshed) | [Terrain viewshed](#Terrain-viewshed)

Standard imports plus `viewshed` via the xrspatial accessor.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

import xrspatial

## Simple viewshed

Start with a synthetic peak from a 2D normal distribution, place an observer off to one side, and check what's visible.

In [ ]:
W = 800
H = 600

OBSERVER_X = -12.5
OBSERVER_Y = 10

x_range = (-20, 20)
y_range = (-20, 20)

rng = np.random.default_rng(42)
normal_df = pd.DataFrame({
   'x': rng.normal(0.5, 1, 10_000_000),
   'y': rng.normal(0.5, 1, 10_000_000),
})

counts, xedges, yedges = np.histogram2d(
    normal_df['x'].values, normal_df['y'].values,
    bins=[W, H], range=[list(x_range), list(y_range)],
)
normal_agg = xr.DataArray(
    counts.T.astype('float64'),
    dims=['y', 'x'],
    coords={
        'y': (yedges[:-1] + yedges[1:]) / 2,
        'x': (xedges[:-1] + xedges[1:]) / 2,
    },
)

normal_illuminated = normal_agg.xrs.hillshade()

fig, ax = plt.subplots(figsize=(10, 7.5))
normal_illuminated.plot.imshow(ax=ax, cmap='gray', alpha=0.5, add_colorbar=False)
ax.scatter([OBSERVER_X], [OBSERVER_Y], c='orange', s=100, zorder=5, edgecolors='white')
ax.set_title('Gaussian peak with observer location')
ax.set_axis_off()
plt.tight_layout()

Viewshed marks each cell as visible or not. Red cells have an unblocked line of sight to the observer.

In [ ]:
view = normal_agg.xrs.viewshed(x=OBSERVER_X, y=OBSERVER_Y)

visible = view.where(view >= 0)

fig, ax = plt.subplots(figsize=(10, 7.5))
normal_illuminated.plot.imshow(ax=ax, cmap='gray', alpha=0.5, add_colorbar=False)
visible.plot.imshow(ax=ax, cmap=ListedColormap(['red']), alpha=0.5, add_colorbar=False)
ax.scatter([OBSERVER_X], [OBSERVER_Y], c='orange', s=100, zorder=5, edgecolors='white')
ax.legend(handles=[Patch(facecolor='red', alpha=0.5, label='Visible'),
                   Patch(facecolor='gray', label='Hidden')],
          loc='lower right', fontsize=11, framealpha=0.9)
ax.set_title('Viewshed from observer (simple peak)')
ax.set_axis_off()
plt.tight_layout()

The peak throws a shadow behind it. Everything on the far side of the ridge is hidden from the observer.

## Terrain viewshed

Same idea on more realistic terrain. Generate a mountain range and place an observer at the center. The `observer_elev` parameter sets the observer's height above ground in meters -- 5 m here, about standing height on a small platform.

In [ ]:
terrain = xr.DataArray(np.zeros((H, W)))
terrain = terrain.xrs.generate_terrain(
    x_range=(-250, 250), y_range=(-250, 250),
    zfactor=6000, warp_strength=0.4,
)

illuminated = terrain.xrs.hillshade()

OBSERVER_X = 0.0
OBSERVER_Y = 0.0

fig, ax = plt.subplots(figsize=(10, 7.5))
illuminated.plot.imshow(ax=ax, cmap='gray', alpha=0.5, add_colorbar=False)
terrain.plot.imshow(ax=ax, cmap='terrain', alpha=0.5, add_colorbar=True,
                    cbar_kwargs={'label': 'Elevation'})
ax.scatter([OBSERVER_X], [OBSERVER_Y], c='orange', s=100, zorder=5, edgecolors='white')
ax.set_title('Generated mountain terrain with observer')
ax.set_axis_off()
plt.tight_layout()

In [ ]:
view = terrain.xrs.viewshed(x=OBSERVER_X, y=OBSERVER_Y, observer_elev=5)

visible = view.where(view >= 0)

fig, ax = plt.subplots(figsize=(10, 7.5))
illuminated.plot.imshow(ax=ax, cmap='gray', alpha=0.5, add_colorbar=False)
terrain.plot.imshow(ax=ax, cmap='terrain', alpha=0.5, add_colorbar=False)
visible.plot.imshow(ax=ax, cmap=ListedColormap(['fuchsia']), alpha=0.5, add_colorbar=False)
ax.scatter([OBSERVER_X], [OBSERVER_Y], c='orange', s=100, zorder=5, edgecolors='white')
ax.legend(handles=[Patch(facecolor='fuchsia', alpha=0.5, label='Visible'),
                   Patch(facecolor='gray', label='Hidden')],
          loc='lower right', fontsize=11, framealpha=0.9)
ax.set_title('Viewshed on mountain terrain (observer_elev=5 m)')
ax.set_axis_off()
plt.tight_layout()

# Save preview image
import pathlib
pathlib.Path('images').mkdir(exist_ok=True)
fig.savefig('images/viewshed_preview.png', bbox_inches='tight', dpi=120)

Fuchsia marks cells visible from the observer. The gaps behind ridges are blind spots where terrain blocks the line of sight. Lower `observer_elev` values create more blind spots.

<div class="alert alert-block alert-warning">
<b>Observer coordinates must match the raster CRS.</b> The <code>x</code> and <code>y</code> parameters are in the same coordinate system as the DataArray's <code>x</code> and <code>y</code> coordinates. If your DEM uses projected coordinates (meters), pass the observer location in meters. If it uses geographic coordinates (degrees), pass longitude and latitude.
</div>

### References

- [Viewshed analysis (Wikipedia)](https://en.wikipedia.org/wiki/Viewshed_analysis)
- Franklin, W. R., & Ray, C. (1994). [Higher isn't Necessarily Better: Visibility Algorithms and Experiments](https://doi.org/10.1007/3-540-58795-0_33). *Advances in GIS Research*.
- [xrspatial.viewshed API docs](https://xarray-spatial.readthedocs.io/en/latest/reference/_autosummary/xrspatial.viewshed.html)